# Diabetic Retinopathy using SVM

## 1. Import Library

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
import joblib
import pickle

## 2. Load Data

In [2]:
with open("../../../data/processed/aptos/sample_50/sample_dataset.pkl", "rb") as f: #[cite: 1]
    data = pickle.load(f)

In [3]:
with open("../../../data/processed/aptos/test_images/sample_dataset.pkl", "rb") as f:
    data_test = pickle.load(f)

## 3. Data Preprocessing

In [4]:
X = data['data']
y = data['labels']
class_names = data['class_names']
IMG_SIZE = X.shape[1]

In [5]:
N = X.shape[0]
X_flatten = X.reshape(N, -1) # Chuyển từ (N, 300, 300, 3) thành (N, 270000)
print(f"Kích thước sau khi duỗi: {X_flatten.shape}")

Kích thước sau khi duỗi: (250, 270000)


In [6]:
NUM_CLASSES = len(class_names)
x_train, x_val, y_train, y_val = train_test_split(
    X_flatten,y, test_size=0.2, random_state=42, stratify=y
)
print(f'Số mẫu: {len(y_train)} | Số lớp: {NUM_CLASSES} | Kích thước ảnh: {IMG_SIZE}x{IMG_SIZE}')

Số mẫu: 200 | Số lớp: 5 | Kích thước ảnh: 300x300


In [7]:
x_test = data_test['data']
y_test = data_test['labels']
print(f'Số mẫu: {len(y_test)} | Số lớp: {NUM_CLASSES} | Kích thước ảnh: {IMG_SIZE}x{IMG_SIZE}')

Số mẫu: 366 | Số lớp: 5 | Kích thước ảnh: 300x300


In [8]:
X_flatten_test = x_test.reshape(N, -1)

## 3. Khởi tạo và huấn luyện mô hình SVM

In [9]:
print("\nBắt đầu huấn luyện mô hình SVM (Có thể mất một lúc)...")
start_time = time.time()


Bắt đầu huấn luyện mô hình SVM (Có thể mất một lúc)...


In [10]:
# ======================
# 1. Khởi tạo Pipeline cho ẢNH
# ======================
pipeline_svm_image = Pipeline([
    # Bước 1: Chuẩn hóa dữ liệu (BẮT BUỘC cho SVM để hội tụ nhanh)
    # Đưa các giá trị pixel từ dải [0, 255] về phân phối chuẩn (mean=0, variance=1)
    ("scaler", StandardScaler()),
    
    # Bước 2: Giảm chiều dữ liệu bằng PCA
    # Giữ lại số lượng thành phần chính giải thích được 95% phương sai của ảnh
    # Giúp giảm từ 270.000 chiều xuống còn vài trăm chiều, SVM sẽ chạy nhanh gấp chục lần
    ("pca", PCA(n_components=0.95, random_state=42)),
    
    # Bước 3: Mô hình phân loại SVM
    ("svm", SVC(
        kernel='rbf',            # Với ảnh, RBF (Radial Basis Function) thường bám biên tốt hơn Linear
        C=1.0,                   # Độ phạt (điều chỉnh nếu bị overfitting/underfitting)
        class_weight='balanced', # Xử lý mất cân bằng 5 class của dữ liệu y tế
        random_state=42
    ))
])

# ======================
# 2. Cross-validation (StratifiedKFold)
# ======================
# n_splits=5 nghĩa là chia tập train thành 5 phần (K=5)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Đang chạy 5-Fold Cross-validation (Sẽ mất thời gian do PCA và SVM tính toán)...")
scores = cross_val_score(
    pipeline_svm_image,
    x_train,       # Truyền mảng đã duỗi phẳng
    y_train,            # Nhãn dạng số nguyên (0 -> 4)
    cv=skf,
    scoring="f1_macro", # Dùng macro f1 để đánh giá công bằng các nhãn hiếm
    n_jobs=-1           # Dùng toàn bộ nhân CPU
)

print("\nF1 từng fold:", scores)
print(f"F1 trung bình (Cross-validation): {np.mean(scores):.4f}")

Đang chạy 5-Fold Cross-validation (Sẽ mất thời gian do PCA và SVM tính toán)...

F1 từng fold: [0.42153846 0.36200531 0.42662679 0.48539826 0.39094239]
F1 trung bình (Cross-validation): 0.4173


In [ ]:
# ======================
# 3. Train FINAL MODEL
# ======================
print("\nĐang huấn luyện mô hình FINAL trên toàn bộ tập Train...")
pipeline_svm_image.fit(x_train, y_train)


Đang huấn luyện mô hình FINAL trên toàn bộ tập Train...


In [ ]:
end_time = time.time()
print(f"✅ Đã huấn luyện xong! Thời gian: {end_time - start_time:.2f} giây")

## 4. Đánh giá mô hình

In [ ]:
print("\nĐang dự đoán trên tập Test...")
y_pred = pipeline_svm_image.predict(X_flatten_test)

print("\n=== TEST RESULT (SVM FOR IMAGES) ===")
print(classification_report(y_test, y_pred))

# ======================
# 5. Lưu Model
# ======================
joblib.dump(pipeline_svm_image, "../../../data/models/dr_svm_pipeline_aptos_50.pkl")
print("✅ Đã lưu toàn bộ Pipeline (Scaler + PCA + SVM) vào 1 file duy nhất!")

## 5. Xem xét kết quả huấn luyện

In [ ]:
# tạo confusion_matrix
cm = confusion_matrix(y_test, y_pred)

# vẽ heatmap
sns.heatmap(cm, annot = True, cmap = "RdPu", fmt = 'd')

# thêm tiêu đề và nhãn trục
plt.title("Confusion Matrix")
plt.xlabel("Dự đoán")
plt.ylabel("Thực tế")

# hiển thị biểu đồ
plt.show()